In [1]:
print('hello !')

hello !


deciding architecture ---

user query --> classifier llm ---> classifies into one of [order_agent, refund_agent, payment_agent,
human_agent(this will be handoff)]

questions like --
i want a refund on thisn ---> for this make a method as a tool taking some rules and replies yes or no
I'm not able to pay ----> maybe an error code associated with the order placed
maybe a discount agent as well? ---> not sure but provides a random discount code to give 5-10% off on products
if query not related to any of this, then hand it over to human agent


classifier is easy

lets says user asks- where is my order - order123

instead of RAG, we SHOULD use DB lookup here. 

so, gpt suggests creating mock data for our usecase here. that is create mock json/python data and use get_order method as a tool in our order agent probably. more clarity to come as we start building this
---but this question is too vague, a real wont ask this question maybe ----need to think about this

maybe since this is V1, we are building the most basic things

user query --> classifier llm ---> classifies into one of [order_agent, refund_agent, payment_agent,
human_agent(this will be handoff)]

lets think abotu the jsom/python/dbn mock of orders :-
orders = {
    order123 = {
        orderDate : ...,
        total : ...,
        payment = {
            paymentStatus : failed/success/something else,
            statusCode : 
        },
        orderStatus : shipped/delivered/failed,
    }
}



GRAPH NODES

  query
    |
classifier
    |
1. order  2. refund                     3.payment    4.Human
    |          |                            |           |
  DBlookup    refundEligibleCheck()       dblookup      handoffToHumanAgwnrt(not sure on thjis)

Let's START !

In [2]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict) :
    messages : Annotated[list, add_messages]
    humanRequired : bool
    category : str

In [3]:
import json

with open('orders.json', 'r', encoding='utf-8') as order:
    orders = json.load(order)


In [4]:
from typing import Literal
from pydantic import BaseModel

#structured output for classifier llm
class classifier(BaseModel) : 
    category : Literal['order', 'refund', 'payment', 'human']

In [5]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

classifierLLM = ChatOpenAI(model='gpt-5-nano')
classifierLLM_with_SO = classifierLLM.with_structured_output(classifier)

In [6]:
#classifier agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

def classifierAgent(old_state : State) :
    PROMPT = f''' 
    You are a classifier agent for an ecommerce platform. Your work is to analyze the incoming user queries and decide
    the category which they belong to. Your response should be the category the query belongs to.
    Here's the user query - {old_state['messages'][-1].content}
    '''
    response = classifierLLM_with_SO.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
    print(f'category : {response.category}')
    return {
        'category' : response.category
    }
    

In [7]:
def classify_route(old_state : State) :
    return 'order' if old_state['category']=='order' else 'refund' if old_state['category']=='refund' else 'payment' if old_state['category'] == 'payment' else 'human'

In [8]:
# check_refund_eligibility tool
from datetime import datetime
from langchain_core.tools import Tool

def check_refund_eligibility(order_id) :
    '''use this to check if a order is eligible for refund or not'''
    if order_id in orders:
        order = orders.get(order_id)
        
        order_date_str = order['order_date']  # Example: "2026-05-29"
        order_date = datetime.strptime(order_date_str, "%Y-%m-%d")
        current_date = datetime.now()
        days_passed = (current_date - order_date).days
        
        if order['refund']['eligible'] and days_passed<=10 and order['shipping']['order_status'] != 'cancelled' :
            return 'item is eligible for a refund'
        else:
            reason = order['refund']['reason'] if not order['refund']['eligible'] else 'More than 10 days since order got placed' if days_passed>10 else order['shipping']['order_status']
            return reason
        
check_refund_eligibility_tool =  Tool(
    name='check_refund_eligibility',
    func=check_refund_eligibility,
    description='use this to check if a order is eligible for refund or not'
)

refund_tools = [check_refund_eligibility_tool]

refund_llm = ChatOpenAI(model='gpt-5-nano')
refund_llm_with_tools = refund_llm.bind_tools(refund_tools)

In [9]:
from langchain_core.tools import Tool

def get_order(order_id) :
    '''Use this to fetch order details from the mock DB'''
    print('Tool Called !')
    if order_id in orders:
        orderDetails = orders.get(order_id)
        # print(f'found order details - {orderDetails}')
        return orderDetails
    else:
        return 'order not found'

orderTool = Tool(
    name='fetch_order_details',
    func=get_order,
    description='Use this to fetch order details from the mock DB'
)

order_related_tools = [orderTool]

worker_llm = ChatOpenAI(model='gpt-5-nano')
worker_llm_w_tools = worker_llm.bind_tools(order_related_tools)


In [10]:
from langchain_community.utilities import GoogleSerperAPIWrapper

serper = GoogleSerperAPIWrapper()

serperTool = Tool(
    name='serper',
    func=serper.run,
    description='Use this to make web searches'
)

web_tools = [serperTool]

human_llm = ChatOpenAI(model='gpt-5-nano')
human_llm_with_tools = human_llm.bind_tools(web_tools)

In [ ]:
from langchain_core.messages import ToolMessage


def orderAgent(old_state:State) :
    PROMPT = f''' 
    You are an Order Support Agent for an e-commerce platform.

    User Query:
    {old_state['messages'][-1].content}

    Instructions:

    * Use the fetch_order_details tool whenever order information is required.
    * Answer ONLY using information returned by the tool.
    * Do not invent tracking details, delivery dates, statuses, or actions by yourself.
    * If the order is not found, respond with a short humorous message.
    * Keep responses concise and customer-friendly.
    If a ToolMessage is already present in the conversation history,
    use that information to answer the user.

    Do not call fetch_order_details again once order information has been retrieved.
    If a ToolMessage is already present in the conversation history,
    DO NOT call any tool again.

    Use the ToolMessage content to answer the user directly.
    
    When order information is available:

    - First identify what the user is asking.
    - Answer ONLY that question.
    - DO NOT DUMP ALL order fields.
    - Summarize and only give the relevant information in plain ENGLISH in one or two sentences.
    
    Examples :
    query - where is my order
    response - your order has been delivered on 13th June 2026.
    
    query - why my payment got failed?
    response - your payment got failed due to this error code - ABC123. Please check details and try again

    '''
    
    if isinstance(old_state["messages"][-1], ToolMessage):
        orderResponse = worker_llm.invoke(
            input=[SystemMessage(content=PROMPT)]
        )
    else:
        orderResponse = worker_llm_w_tools.invoke(
            input=[SystemMessage(content=PROMPT)]
        )
    
    return {
        'messages' : [orderResponse]
    }

In [ ]:
def refund_agent(old_state) :
    PROMPT = f''' 
    You are a Refund Support Agent for an e-commerce platform.

    User Query:
    {old_state['messages'][-1].content}

    Instructions:

    * Use the check_refund_eligibility tool to determine refund eligibility.
    * Base your response only on the tool output.
    * If the order is eligible, clearly inform the customer.
    * If the order is not eligible, explain the reason returned by the tool.
    * Do not invent refund policies or business rules.
    * Keep responses concise and professional.
    
    When order information is available:

    - First identify what the user is asking.
    - Answer ONLY that question.
    - Do not dump all order fields.
    - Summarize only the relevant information.

    Examples:

    User: Where is my order?
    Answer: Your order has been delivered on 2026-05-22.

    User: What is my payment status?
    Answer: Your payment was successful.

    User: Is my order delivered?
    Answer: Yes, your order was delivered on 2026-05-22.
    '''
    refund_response = refund_llm_with_tools.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
        
    return {
        'messages' : [refund_response]
    }

In [ ]:
def paymentAgent(old_state) :
    PROMPT = f''' 
    You are a Payment Support Agent for an e-commerce platform.

    User Query:
    {old_state['messages'][-1].content}

    Instructions:

    * Use the fetch_order_details tool whenever payment information is needed.
    * Answer only using information returned by the tool.
    * Explain payment status clearly and concisely.
    * Do not invent payment actions, refunds, retries, or support processes.
    * If the order is not found, clearly inform the customer.
    * Keep responses short and professional.
    If a ToolMessage is already present in the conversation history,
    use that information to answer the user.

    Do not call fetch_order_details again once order information has been retrieved.
    If a ToolMessage is already present in the conversation history,
    DO NOT call any tool again.

    Use the ToolMessage content to answer the user directly.
    
    When order information is available:

    - First identify what the user is asking.
    - Answer ONLY that question.
    - DO NOT DUMP ALL order fields.
    - Summarize and only give the relevant information in plain ENGLISH in one or two sentences.

    Examples:

    User: Where is my order?
    Answer: Your order has been delivered on 2026-05-22.

    User: What is my payment status?
    Answer: Your payment was successful.

    User: Is my order delivered?
    Answer: Yes, your order was delivered on 2026-05-22.
    '''
    payment_response = worker_llm_w_tools.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
    
    return {
        'messages' : [payment_response]
    }

In [51]:
def humanAgent(old_state : State) : 
    return {
        'messages' : [
            AIMessage(content=''' 
Your request has been escalated to a human support representative.

Ticket ID: TKT-12345

Expected response time: 24 hours.

''')
        ]
    }

In [52]:
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

graph_builder = StateGraph(State)
graph_builder.add_node('classifier', classifierAgent)
graph_builder.add_node('orderAgent', orderAgent)
graph_builder.add_node('fetchOrderTool', ToolNode(tools=order_related_tools))
graph_builder.add_node('refundTools', ToolNode(tools=refund_tools))
graph_builder.add_node('paymentTool', ToolNode(tools=order_related_tools))
graph_builder.add_node('web_searchTool', ToolNode(tools=web_tools))
graph_builder.add_node('refundAgent', refund_agent)
graph_builder.add_node('paymentAgent', paymentAgent)
graph_builder.add_node('humanAgent', humanAgent)

graph_builder.add_edge(START, 'classifier')
graph_builder.add_conditional_edges('classifier', classify_route,{'order' : 'orderAgent', 'refund' : 'refundAgent', 'payment' : 'paymentAgent', 'human':'humanAgent'})
graph_builder.add_conditional_edges('orderAgent', tools_condition, {'tools':'fetchOrderTool','__end__' : END})
graph_builder.add_conditional_edges('refundAgent', tools_condition, {'tools':'refundTools','__end__' : END})
graph_builder.add_conditional_edges('paymentAgent', tools_condition, {'tools':'paymentTool','__end__' : END})
graph_builder.add_conditional_edges('humanAgent', tools_condition, {'tools':'web_searchTool','__end__' : END})
graph_builder.add_edge('fetchOrderTool', 'orderAgent')
graph_builder.add_edge('refundTools', 'refundAgent')
graph_builder.add_edge('paymentTool', 'paymentAgent')
graph_builder.add_edge('web_searchTool', 'humanAgent')

graph = graph_builder.compile()


In [53]:
from IPython.display import display, Image
#display(Image(graph.get_graph().draw_mermaid_png()))

In [54]:
import os
print("LangSmith Tracing Enabled:", os.getenv("LANGSMITH_TRACING"))

LangSmith Tracing Enabled: true


In [60]:
#ORDER QUERY
state = {
    'messages' : [HumanMessage(content='When will ORD1008 arrive?')]
}
res = graph.invoke(state)
print(res['messages'][-1].content)

category : order
Tool Called !
Here’s the current status of your order:

- Shipping: Cancelled (tracking and delivery details not available)
- Payment: Success (Debit Card) | Transaction ID TXN1008
- Refund: Not Requested; Reason: Cancelled by customer; Eligible: false

What would you like to do next? I can help you place a new order or review any refund options if needed.


In [ ]:
#REFUND QUERY
state = {
    'messages' : [HumanMessage(content='is my order ORD1005 for a refund?')]
}
res = graph.invoke(state)
print(res['messages'][-1].content)

In [57]:
# #PAYMENT QUERY
# state = {
#     'messages' : [HumanMessage(content='why payment of my order ORD1004 got failed? reason?')]
# }
# res = graph.invoke(state)
# print(res['messages'][-1].content)

In [58]:
# state = {
#     'messages' : [HumanMessage(content='Has ORD1008 been delivered')]
# }
# res = graph.invoke(state)
# print(res['messages'][-1].content)